In [20]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import requests

raw_data_path = '../data/raw'

# despesas-pela-cota-para-exercício-da-atividade-parlamentar

In [21]:
import os
import zipfile
import requests

anos = [2023, 2024, 2025, 2026]

for ano in anos:
  url = f'https://www.camara.leg.br/cotas/Ano-{ano}.csv.zip'
  nome_zip = f'{raw_data_path}/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-{ano}.csv.zip'

  print(f'Baixando o arquivo de {ano}: {url}')

  try:
    response = requests.get(url, stream=True, timeout=60)

    if response.status_code == 200:
      print(f'{raw_data_path}/{nome_zip}')
      with open(nome_zip, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB por chunk
          if chunk:
            f.write(chunk)
      print(f'-> Download concluído: {nome_zip}')

    else:
      print(
          f'-> Falha ao baixar o ano {ano}. Status HTTP:'
          f' {response.status_code}\n'
      )

  except requests.exceptions.RequestException as e:
    print(f'-> Erro de conexão ao baixar o ano {ano}: {e}\n')

Baixando o arquivo de 2023: https://www.camara.leg.br/cotas/Ano-2023.csv.zip
../data/raw/../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2023.csv.zip
-> Download concluído: ../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2023.csv.zip
Baixando o arquivo de 2024: https://www.camara.leg.br/cotas/Ano-2024.csv.zip
../data/raw/../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2024.csv.zip
-> Download concluído: ../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2024.csv.zip
Baixando o arquivo de 2025: https://www.camara.leg.br/cotas/Ano-2025.csv.zip
../data/raw/../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2025.csv.zip
-> Download concluído: ../data/raw/despesas-pela-cota-para-exercício-da-atividade-parlamentar/Ano-2025.csv.zip
Baixando o arquivo de 2026: https://www.camara.leg.br/cotas/Ano-2026.csv.zip
../data/raw/../data/raw/despesas-pela-cota-para-exercício-d

# Partidos

In [22]:
import json
import requests

url = "https://dadosabertos.camara.leg.br/api/v2/partidos"
params = {"itens": "1000", "ordem": "ASC", "ordenarPor": "sigla"}
headers = {"accept": "application/json"}

response = requests.get(url, params=params, headers=headers)

if response.status_code == 200:
  dados = response.json()

  with open(f"{raw_data_path}/partidos/partidos.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=4)

  print("Download realizado com sucesso!")
else:
  print(f"Erro na requisição: {response.status_code}")

Download realizado com sucesso!


# Deputados

In [23]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

options = Options()

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)
wait = WebDriverWait(driver, 15)

try:
  # 1. Acessa a página da Base dos Dados
  url = "https://basedosdados.org/dataset/3d388daa-2d20-49eb-8f55-6c561bef26b6?table=d37931b4-8b67-47c7-b910-014695dc8f36"
  driver.get(url)
  print("Página acessada. Aguardando o carregamento...")

  # 2. Localiza o botão "Download"
  print('Procurando a aba/botão "Download"...')
  botao_guia_download = wait.until(
      EC.presence_of_element_located(
          (By.XPATH, "//button[normalize-space()='Download']")
      )
  )

  # Rola a página até o elemento para garantir que ele esteja visível na tela
  driver.execute_script(
      "arguments[0].scrollIntoView({block: 'center'});", botao_guia_download
  )
  time.sleep(1)

  # Clica via JavaScript (contorna qualquer sobreposição na tela)
  driver.execute_script("arguments[0].click();", botao_guia_download)
  print("Guia de download aberta com sucesso!")

  # 3. Pequena pausa para a nova seção/guia renderizar na tela
  time.sleep(3)

  # 4. Localiza e clica no segundo botão (Download da tabela) usando o seu XPath
  print("Procurando o botão de download da tabela...")
  botao_tabela = wait.until(
      EC.presence_of_element_located(
          (
              By.XPATH,
              "/html/body/div[1]/div/div[1]/div/div[4]/div[2]/div[3]/div[4]/div[2]/div/div[2]/div[2]/div/button",
          )
      )
  )

  # Rola a tela até o botão da tabela e clica via JS também, por segurança
  driver.execute_script(
      "arguments[0].scrollIntoView({block: 'center'});", botao_tabela
  )
  time.sleep(1)
  driver.execute_script("arguments[0].click();", botao_tabela)
  print("Botão de download da tabela clicado com sucesso!")

  # 5. Tempo para o arquivo começar a baixar
  time.sleep(10)

except Exception as e:
  print(f"Ocorreu um erro durante a execução: {e}")

finally:
  driver.quit()

Página acessada. Aguardando o carregamento...
Procurando a aba/botão "Download"...
Guia de download aberta com sucesso!
Procurando o botão de download da tabela...
Botão de download da tabela clicado com sucesso!


# Proposição

In [24]:
anos = [2023, 2024, 2025, 2026]

for ano in anos:
  url = f'https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-{ano}.csv'
  nome_zip = f'{raw_data_path}/proposicoes/proposicoes-{ano}.csv'

  print(f'Baixando o arquivo de {ano}: {url}')

  try:
    response = requests.get(url, stream=True, timeout=60)

    if response.status_code == 200:
      print(f'{raw_data_path}/{nome_zip}')
      with open(nome_zip, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB por chunk
          if chunk:
            f.write(chunk)
      print(f'-> Download concluído: {nome_zip}')

    else:
      print(
          f'-> Falha ao baixar o ano {ano}. Status HTTP:'
          f' {response.status_code}\n'
      )

  except requests.exceptions.RequestException as e:
    print(f'-> Erro de conexão ao baixar o ano {ano}: {e}\n')

Baixando o arquivo de 2023: https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-2023.csv
../data/raw/../data/raw/proposicoes/proposicoes-2023.csv
-> Download concluído: ../data/raw/proposicoes/proposicoes-2023.csv
Baixando o arquivo de 2024: https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-2024.csv
../data/raw/../data/raw/proposicoes/proposicoes-2024.csv
-> Download concluído: ../data/raw/proposicoes/proposicoes-2024.csv
Baixando o arquivo de 2025: https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-2025.csv
../data/raw/../data/raw/proposicoes/proposicoes-2025.csv
-> Download concluído: ../data/raw/proposicoes/proposicoes-2025.csv
Baixando o arquivo de 2026: https://dadosabertos.camara.leg.br/arquivos/proposicoes/csv/proposicoes-2026.csv
../data/raw/../data/raw/proposicoes/proposicoes-2026.csv
-> Download concluído: ../data/raw/proposicoes/proposicoes-2026.csv


# Proposição Autores

In [25]:
anos = [2023, 2024, 2025, 2026]

for ano in anos:
  url = f'https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-{ano}.csv'
  nome_zip = f'{raw_data_path}/proposicoes-autores/proposicoesAutores-{ano}.csv'

  print(f'Baixando o arquivo de {ano}: {url}')

  try:
    response = requests.get(url, stream=True, timeout=60)

    if response.status_code == 200:
      print(f'{raw_data_path}/{nome_zip}')
      with open(nome_zip, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1 MB por chunk
          if chunk:
            f.write(chunk)
      print(f'-> Download concluído: {nome_zip}')

    else:
      print(
          f'-> Falha ao baixar o ano {ano}. Status HTTP:'
          f' {response.status_code}\n'
      )

  except requests.exceptions.RequestException as e:
    print(f'-> Erro de conexão ao baixar o ano {ano}: {e}\n')

Baixando o arquivo de 2023: https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-2023.csv
../data/raw/../data/raw/proposicoes-autores/proposicoesAutores-2023.csv
-> Download concluído: ../data/raw/proposicoes-autores/proposicoesAutores-2023.csv
Baixando o arquivo de 2024: https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-2024.csv
../data/raw/../data/raw/proposicoes-autores/proposicoesAutores-2024.csv
-> Download concluído: ../data/raw/proposicoes-autores/proposicoesAutores-2024.csv
Baixando o arquivo de 2025: https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-2025.csv
../data/raw/../data/raw/proposicoes-autores/proposicoesAutores-2025.csv
-> Download concluído: ../data/raw/proposicoes-autores/proposicoesAutores-2025.csv
Baixando o arquivo de 2026: https://dadosabertos.camara.leg.br/arquivos/proposicoesAutores/csv/proposicoesAutores-2026.csv
../data/raw/../data/raw/proposicoes-autores